# 01 — MuJoCo Playground で四脚（Go1）の歩行を学習する

テスト機 `Go1JoystickFlatTerrain` を Brax PPO で学習し、歩く動画を出すまで。

**準備**: ランタイム → ランタイムのタイプを変更 → **GPU（T4 以上）**。

ロジックは `quadleg_rl/train.py` にあり、このノートブックは呼び出すだけ。コードの編集は Zed で行い、GitHub に push → 下のセルで pull。

In [ ]:
#@title 1. GPU 確認
!nvidia-smi -L

In [ ]:
#@title 2. リポジトリ取得（GitHub に push 済みの URL を入れる。未 push なら左のファイル欄に quadleg_rl/ フォルダをアップロード）
REPO_URL = "https://github.com/yosihitoyasudasub/quadleg-rl.git"  #@param {type:"string"}
REPO_DIR = "/content/quadleg-rl"
import os, sys
if REPO_URL and "<your-name>" not in REPO_URL:
    if os.path.isdir(REPO_DIR):
        !cd $REPO_DIR && git pull
    else:
        !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR
else:
    print("REPO_URL 未設定。/content に quadleg_rl/ を置いてください。")
    REPO_DIR = "/content"
    %cd /content
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# clone 前にパスを sys.path に入れていると importlib が「中身なし」をキャッシュしているので破棄する
import importlib
importlib.invalidate_caches()
import quadleg_rl
print("OK:", quadleg_rl.__file__)

In [ ]:
#@title 3. インストール（初回 3〜4 分）
# brax は PyPI 最新の 0.14.2（2026-03-15）が jax.device_put_replicated を使っており、
# この API は JAX 0.10 で削除済みのため PPO 初期化時に AttributeError になる。
# main（2026-03-25 のコミット）で jax.device_put に修正済みだが未リリースなので GitHub から入れる。
# flax も Colab のプリインストール版が古く、jax.core.get_opaque_trace_state を直接呼ぶため更新する。
%pip install -q -U "jax[cuda12]" playground mediapy "flax>=0.12"
%pip install -q -U "brax @ git+https://github.com/google/brax.git@main"
import jax, flax, brax, mujoco
print("jax", jax.__version__, "| flax", flax.__version__, "| brax", brax.__version__, "| mujoco", mujoco.__version__)
print("backend:", jax.default_backend(), jax.devices())
import inspect, brax.training.pmap as _p
print("brax pmap patched:", "device_put_replicated" not in inspect.getsource(_p))   # True なら OK
# 「RESTART SESSION」を促された場合は再起動し、セル 2 と 4 を実行し直してからセル 5 へ

In [ ]:
#@title 4. Google Drive をマウント（チェックポイント・動画の保存先）
from google.colab import drive
drive.mount("/content/drive")
LOGDIR = "/content/drive/MyDrive/quadleg-rl/logs"
import os; os.makedirs(LOGDIR, exist_ok=True); print(LOGDIR)

In [ ]:
#@title 5. 学習（T4: 2,000 万ステップで 20〜40 分。まず短く回して流れを確認）
# セル 3 のインストール後にランタイムが再起動すると cwd と sys.path が戻るので、ここで通し直す
import os, sys
REPO_DIR = globals().get("REPO_DIR", "/content/quadleg-rl")
if not os.path.isdir(os.path.join(REPO_DIR, "quadleg_rl")):
    raise RuntimeError(f"{REPO_DIR}/quadleg_rl が無い。セル 2 を実行（再起動後なら再実行）してください。")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import importlib
importlib.invalidate_caches()
if "LOGDIR" not in globals():
    raise RuntimeError("LOGDIR 未定義。セル 4（Drive マウント）を実行してください。")

ENV_NAME = "Go1JoystickFlatTerrain"  #@param ["Go1JoystickFlatTerrain", "Go1JoystickRoughTerrain", "Go1Getup", "SpotFlatTerrainJoystick", "BarkourJoystick"]
NUM_TIMESTEPS = 20_000_000  #@param {type:"integer"}
NUM_EVALS = 10  #@param {type:"integer"}

from quadleg_rl import train
result = train.train(ENV_NAME, num_timesteps=NUM_TIMESTEPS, num_evals=NUM_EVALS, logdir=LOGDIR)
train.plot_history(result)

In [ ]:
#@title 6. 歩行動画（前進 0.5 m/s の指令を固定）
import mediapy as media
path = train.render_video(result, result.logdir / "rollout_forward.mp4", command=(0.5, 0.0, 0.0))
media.show_video(media.read_video(path), fps=25)

In [ ]:
#@title 7. 旋回動画（ヨー 1 rad/s）
path = train.render_video(result, result.logdir / "rollout_turn.mp4", command=(0.0, 0.0, 1.0))
media.show_video(media.read_video(path), fps=25)

## 次のステップ

- `NUM_TIMESTEPS` を 1 億にすると公式設定と同じ（T4 では数時間。Colab Pro の L4/A100 推奨）。
- 切断されたら、セル 5 で `train.train(..., restore_checkpoint_path=f"{LOGDIR}/<run>/checkpoints")` で再開。
- 自分の脚（XL-320/XL330 四脚）に差し替えるには `quadleg_rl/` に MJCF と環境クラスを追加する（02 ノートブック予定）。